# Notebook 04's tlrn fits on a Colab TPU

This notebook trains the slow part of `analysis/04_nn_architectures_vs_classical.ipynb` (the tlrn fits) with ibnr's optional JAX backend, on a Colab accelerator runtime, and saves every fit to Google Drive as soon as it finishes. It does not run the rest of notebook 04: copy the saved fits back and load them there (last cell).

**Before you rely on it.** The JAX backend has been tested on a CPU only. The first run of this notebook on a TPU is what measures the speed, and the `--smoke` step below is the cheap check that everything is wired. The backend, the `[jax]` extra and the runner script exist on `main` only after the pull request that adds them is merged; the install cell says which source it uses.

Set the runtime first: Runtime, Change runtime type, TPU (or GPU). See `docs/tlrn-on-colab.md` for the reasoning behind each step.

## 1. Check the runtime

Colab comes with a jax built for its accelerator. Keep it. You want `tpu` (or `gpu`) below; `cpu` means the runtime type was not changed, and the JAX backend would be slower than torch.

In [ ]:
import jax

print(jax.__version__, jax.default_backend(), jax.devices())

## 2. Install

ibnr asks for jax 0.7 or later, so the install must not replace the runtime's own jax. Pick the source: `main` until a release carries the backend, then the plain package.

Colab's runtime is on Python 3.13 and ibnr declares 3.11 to 3.12, because two other extras (`bayesian`, `interop`) cannot install on 3.13. This notebook uses neither, so `--ignore-requires-python` is safe here; the core, `nn` and `jax` parts were tested on 3.13.

In [ ]:
SOURCE = "main"  # "main" until the release carries the JAX backend, then "pypi"

if SOURCE == "main":
    %pip install -q --ignore-requires-python "ibnr[jax] @ git+https://github.com/EKtheSage/ibnr@main" cas-schedule-p==2026.6.13
else:
    %pip install -q --ignore-requires-python "ibnr[jax]" cas-schedule-p==2026.6.13

In [ ]:
# The runner script is not in the wheel; fetch it from main.
!curl -sSLO https://raw.githubusercontent.com/EKtheSage/ibnr/main/scripts/tlrn_colab.py
import jax

import ibnr

print("ibnr", ibnr.__version__, "| jax", jax.__version__, jax.default_backend())

## 3. Keep the fits on Google Drive

A disconnect then loses at most the fit that was running. Run the same command again and saved fits with matching settings are skipped.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/ibnr/tlrn_fits"
OUT_AY = "/content/drive/MyDrive/ibnr/tlrn_fits_ay"  # its own folder: different settings

## 4. Smoke check

Ten epochs, four members. It builds the triangle, checks the cohort against notebook 04's counts and hash, and trains both fits. Run it first; if it fails, stop here. Use its own folder so its files never sit beside the real ones.

In [ ]:
!python tlrn_colab.py --out /content/drive/MyDrive/ibnr/tlrn_fits_smoke --smoke

## 5. The published protocol

Forty members for each of `tlrn_8` (paid only) and `tlrn_13` (paid, incurred and case), all kept. Progress prints every hundred epochs.

In [ ]:
!python tlrn_colab.py --out {OUT}

## 6. The accident-year variant (optional)

The companion study's method: twenty members at each of four valuation dates, eighty in all. This is the run notebook 04 on the laptop is doing on CPU.

In [ ]:
!python tlrn_colab.py --out {OUT_AY} --config accident_year

## 7. What was saved

Each fit has a `.pkl` and a `.json` record of its settings and what it ran on (jax version, device, seconds). The seconds are the TPU timing to report.

In [ ]:
import json
from pathlib import Path

for folder in (OUT, OUT_AY):
    for record in sorted(Path(folder).glob("*.json")):
        info = json.loads(record.read_text(encoding="utf-8"))
        print(record.name, {k: v for k, v in info.items() if k != "settings"})

## 8. Use the fits in notebook 04

Copy the folder to the machine that runs notebook 04, then in its tlrn cell load each entry instead of fitting it. The file names carry the ibnr version that saved them, and an entry must be loaded with that ibnr. A pickle runs code when it is loaded, so load only files you wrote.

```python
import json, pickle
from pathlib import Path
import ibnr

FITS = Path("tlrn_fits")
ROWS = ("tlrn_8", "tlrn_13")  # ("tlrn_8_ay", "tlrn_13_ay") for the accident-year folder
tlrn_fits = {}
for row in ROWS:
    stem = f"{row}-ibnr{ibnr.__version__}"
    with open(FITS / f"{stem}.pkl", "rb") as handle:
        tlrn_fits[row] = pickle.load(handle)
```